# **Kenya Public Health Data Pipeline**
## HIV Treatment Cascade & Health Systems Analytics
### Wayne Willis Omondi

**Architecture:** Extract → Load → Transform (ELT) | **Engine:** DuckDB + Pandas | **Visualization:** Altair

| # | Source | Access Method | Base URL |
|---|--------|---------------|----------|
| 1 | WHO Global Health Observatory | REST API (OData v4) | `https://ghoapi.azureedge.net/api/` |
| 2 | World Bank Open Data | REST API (JSON) | `https://api.worldbank.org/v2/` |
| 3 | UNAIDS Estimates 2026 | Versioned ZIP/CSV | `https://aidsinfo.unaids.org/public/documents/Estimates_2026_en.zip` |
| 4 | The Global Fund Data Service | OData v4.2 | `https://fetch.theglobalfund.org/v4.2/odata/` |

**Scope:** Kenya (focus country) benchmarked against Uganda, Tanzania, Ethiopia and Rwanda · **Period:** 2000–2023

The UNAIDS release is pinned to 2026 for reproducibility. Global Fund transactions retain negative adjustments so annual totals are net disbursements.

## Architecture

```text
WHO GHO API ───────────────┐
World Bank API ────────────┤
UNAIDS versioned ZIP/CSV ──┼─> pandas extraction ─> DuckDB stg_* tables
Global Fund OData v4.2 ────┘                             |
                                                         v
                                                  clean_* tables
                                                         |
                                      +------------------+------------------+
                                      |                                     |
                                      v                                     v
                                  feat_* tables                      dim_* / fact_*
                                      |                                     |
                                      +------------------+------------------+
                                                         v
                                                SQL analysis datasets
                                                         |
                                                         v
                                                Altair visualizations
```


## Prerequisite

In [1]:
!python --version

Python 3.14.6


## 1. Environment Setup

In [2]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "duckdb", "pandas", "requests", "altair", "openpyxl", "--quiet"],
    check=True,
)
print("Dependencies ready.")

Dependencies ready.


In [3]:
import io
import re
import json
import warnings
import requests

import numpy as np
import pandas as pd
import duckdb
import altair as alt

warnings.filterwarnings("ignore")
alt.data_transformers.disable_max_rows()

print(f"DuckDB {duckdb.__version__}  |  pandas {pd.__version__}  |  altair {alt.__version__}")


DuckDB 1.5.5  |  pandas 2.3.3  |  altair 6.2.2


### Database Connection

In [4]:
# Persistent DuckDB connection
DB_PATH = "database/kenya_health_data.duckdb"
con = duckdb.connect(DB_PATH)
con.execute("PRAGMA threads=4")

In [5]:
# Shared constants
EA_ISO3  = ["KEN", "UGA", "TZA", "ETH", "RWA"]
EA_NAMES = [
    "Kenya", "Uganda", "Tanzania",
    "United Republic of Tanzania", "Ethiopia", "Rwanda",
]
YEAR_START, YEAR_END = 2000, 2023

ISO3_MAP = {
    "Kenya": "KEN", "Uganda": "UGA",
    "Tanzania": "TZA", "United Republic of Tanzania": "TZA",
    "Ethiopia": "ETH", "Rwanda": "RWA",
    "Sub-Saharan Africa": "SSA", "Eastern Sub-Saharan Africa": "ESSA",
}

## 2. Data Sources & Ingestion

## a. Source 1: WHO Global Health Observatory

**API:** `https://ghoapi.azureedge.net/api/`

The pipeline uses official WHO indicator metadata to validate every manual code-to-metric mapping before downloading observations. A mismatch stops execution instead of silently assigning the wrong unit or meaning.

| WHO code | Official WHO definition | Normalized field |
|---|---|---|
| `HIV_0000000001` | Estimated number of people (all ages) living with HIV | `plhiv_all_ages` |
| `MDG_0000000029` | Prevalence of HIV among adults aged 15 to 49 (%) | `hiv_prevalence_pct` |
| `HIV_0000000009` | Reported number of people receiving antiretroviral therapy | `on_art_reported` |
| `HIV_ARTCOVERAGE` | Estimated ART coverage among people living with HIV (%) | `art_coverage_pct` |
| `HIV_0000000026` | Number of new HIV infections | `new_infections` |
| `HIV_0000000006` | Number of people dying from HIV-related causes | `hiv_related_deaths` |
| `WHOSIS_000001` | Life expectancy at birth (years) | `life_expectancy` |

WHO uncertainty bounds are retained where supplied. Life expectancy is sex-disaggregated; the feature layer explicitly uses the both-sexes record.

In [6]:
WHO_BASE = "https://ghoapi.azureedge.net/api"

# Each code is paired with both the notebook field and WHO's official metadata label.
# The metadata assertion below prevents a valid-looking but semantically wrong pull.
WHO_INDICATORS = {
    "HIV_0000000001": {
        "label": "plhiv_all_ages",
        "official_name": "Estimated number of people (all ages) living with HIV",
    },
    "MDG_0000000029": {
        "label": "hiv_prevalence_pct",
        "official_name": "Prevalence of HIV among adults aged 15 to 49 (%)",
    },
    "HIV_0000000009": {
        "label": "on_art_reported",
        "official_name": "Reported number of people receiving antiretroviral therapy",
    },
    "HIV_ARTCOVERAGE": {
        "label": "art_coverage_pct",
        "official_name": "Estimated antiretroviral therapy coverage among people living with HIV (%)",
    },
    "HIV_0000000026": {
        "label": "new_infections",
        "official_name": "Number of new HIV infections",
    },
    "HIV_0000000006": {
        "label": "hiv_related_deaths",
        "official_name": "Number of people dying from HIV-related causes",
    },
    "WHOSIS_000001": {
        "label": "life_expectancy",
        "official_name": "Life expectancy at birth (years)",
    },
}

def validate_who_indicator_metadata(indicators: dict) -> pd.DataFrame:
    """Confirm that every configured code still has the intended WHO meaning."""
    code_filter = " or ".join(
        f"IndicatorCode eq '{code}'" for code in indicators
    )
    response = requests.get(
        f"{WHO_BASE}/Indicator",
        params={"$filter": f"Language eq 'EN' and ({code_filter})"},
        timeout=45,
    )
    response.raise_for_status()
    metadata = pd.DataFrame(response.json().get("value", []))
    actual_names = metadata.set_index("IndicatorCode")["IndicatorName"].to_dict()

    mismatches = {
        code: {"expected": spec["official_name"], "actual": actual_names.get(code)}
        for code, spec in indicators.items()
        if actual_names.get(code) != spec["official_name"]
    }
    if mismatches:
        raise ValueError(f"WHO indicator metadata mismatch: {mismatches}")
    return metadata[["IndicatorCode", "IndicatorName"]].sort_values("IndicatorCode")

def fetch_who(indicator_code: str, countries: list, start: int = 2000) -> pd.DataFrame:
    """Fetch one WHO GHO OData indicator for a list of ISO3 country codes."""
    country_filter = " or ".join(f"SpatialDim eq '{code}'" for code in countries)
    response = requests.get(
        f"{WHO_BASE}/{indicator_code}",
        params={"$filter": f"({country_filter}) and TimeDim ge {start}"},
        timeout=45,
    )
    response.raise_for_status()
    records = response.json().get("value", [])
    if not records:
        return pd.DataFrame()
    frame = pd.DataFrame(records)
    frame["indicator_code"] = indicator_code
    return frame

who_metadata = validate_who_indicator_metadata(WHO_INDICATORS)
print("WHO indicator metadata validated:")
print(who_metadata.to_string(index=False))

frames_who = []
for indicator_code, specification in WHO_INDICATORS.items():
    label = specification["label"]
    print(f"  WHO {indicator_code} ({label}) ...", end="  ", flush=True)
    frame = fetch_who(indicator_code, EA_ISO3, start=YEAR_START)
    if frame.empty:
        raise ValueError(f"WHO returned no records for required indicator {indicator_code}")
    frame["indicator_label"] = label
    frame["indicator_official_name"] = specification["official_name"]
    frames_who.append(frame)
    print(f"{len(frame):,} records")

df_who_raw = pd.concat(frames_who, ignore_index=True)
expected_codes = set(WHO_INDICATORS)
actual_codes = set(df_who_raw["IndicatorCode"].unique())
if actual_codes != expected_codes:
    raise ValueError(
        f"WHO indicator coverage mismatch. Expected {expected_codes}; received {actual_codes}"
    )

print(
    f"\nWHO GHO raw: {len(df_who_raw):,} rows | "
    f"{df_who_raw['IndicatorCode'].nunique()} indicators | "
    f"{df_who_raw['SpatialDim'].nunique()} countries"
)
df_who_raw.head(3)

WHO indicator metadata validated:
  IndicatorCode                                                              IndicatorName
 HIV_0000000001                      Estimated number of people (all ages) living with HIV
 HIV_0000000006                             Number of people dying from HIV-related causes
 HIV_0000000009                 Reported number of people receiving antiretroviral therapy
 HIV_0000000026                                               Number of new HIV infections
HIV_ARTCOVERAGE Estimated antiretroviral therapy coverage among people living with HIV (%)
 MDG_0000000029                           Prevalence of HIV among adults aged 15 to 49 (%)
  WHOSIS_000001                                           Life expectancy at birth (years)
  WHO HIV_0000000001 (plhiv_all_ages) ...  130 records
  WHO MDG_0000000029 (hiv_prevalence_pct) ...  130 records
  WHO HIV_0000000009 (on_art_reported) ...  130 records
  WHO HIV_ARTCOVERAGE (art_coverage_pct) ...  130 records
  WHO HIV_

,Id,IndicatorCode,SpatialDimType,SpatialDim,TimeDimType,ParentLocationCode,ParentLocation,Dim1Type,TimeDim,Dim1,...,Low,High,Comments,Date,TimeDimensionValue,TimeDimensionBegin,TimeDimensionEnd,indicator_code,indicator_label,indicator_official_name
0,677227,HIV_0000000001,COUNTRY,ETH,YEAR,AFR,Africa,None,2000,None,...,1000000.0,1500000.0,None,2026-07-31T13:20:25.203+02:00,2000,2000-01-01T00:00:00+01:00,2000-12-31T00:00:00+01:00,HIV_0000000001,plhiv_all_ages,Estimated number of people (all ages) living w...
1,677228,HIV_0000000001,COUNTRY,KEN,YEAR,AFR,Africa,None,2000,None,...,1600000.0,1800000.0,None,2026-07-31T13:20:25.203+02:00,2000,2000-01-01T00:00:00+01:00,2000-12-31T00:00:00+01:00,HIV_0000000001,plhiv_all_ages,Estimated number of people (all ages) living w...
2,677235,HIV_0000000001,COUNTRY,RWA,YEAR,AFR,Africa,None,2000,None,...,220000.0,270000.0,None,2026-07-31T13:20:25.203+02:00,2000,2000-01-01T00:00:00+01:00,2000-12-31T00:00:00+01:00,HIV_0000000001,plhiv_all_ages,Estimated number of people (all ages) living w...


## B. Source 2: World Bank Open Data API

**Base URL:** `https://api.worldbank.org/v2/`  
No authentication. Multi-country with `;`-separated ISO3 codes. Auto-paginated.

| Indicator | Metric |
|-----------|--------|
| `SH.HIV.INCD.ZS` | HIV incidence (per 1,000 uninfected, 15–49) |
| `SH.HIV.ARTC.ZS` | ART coverage (% of PLHIV) |
| `SH.XPD.CHEX.GD.ZS` | Health expenditure (% of GDP) |
| `SH.XPD.CHEX.PC.CD` | Health expenditure per capita (current USD) |
| `SH.MED.NUMW.P3` | Nurses and midwives per 1,000 |
| `SH.MED.PHYS.ZS` | Physicians per 1,000 |
| `SP.POP.TOTL` | Total population |
| `NY.GDP.PCAP.CD` | GDP per capita (current USD) |


In [7]:
WB_BASE = "https://api.worldbank.org/v2"
WB_ISO  = ";".join(EA_ISO3)

WB_INDICATORS = {
    "SH.HIV.INCD.ZS":    "hiv_incidence_per_1000",
    "SH.HIV.ARTC.ZS":    "art_coverage_wb_pct",
    "SH.XPD.CHEX.GD.ZS": "health_exp_pct_gdp",
    "SH.XPD.CHEX.PC.CD": "health_exp_per_cap_usd",
    "SH.MED.NUMW.P3":    "nurses_per_1000",
    "SH.MED.PHYS.ZS":    "physicians_per_1000",
    "SP.POP.TOTL":       "population_total",
    "NY.GDP.PCAP.CD":    "gdp_per_cap_usd",
}

def fetch_wb(indicator: str, countries: str = WB_ISO,
             start: int = YEAR_START, end: int = YEAR_END) -> pd.DataFrame:
    """Paginated World Bank indicator fetch for multiple countries."""
    url    = f"{WB_BASE}/country/{countries}/indicator/{indicator}"
    params = {"format": "json", "per_page": 500, "date": f"{start}:{end}"}
    recs, page = [], 1
    while True:
        params["page"] = page
        r = requests.get(url, params=params, timeout=45)
        r.raise_for_status()
        payload = r.json()
        if len(payload) < 2 or not payload[1]:
            break
        recs.extend(payload[1])
        if page >= payload[0].get("pages", 1):
            break
        page += 1
    if not recs:
        return pd.DataFrame()
    df = pd.json_normalize(recs)
    df["indicator_id"] = indicator
    return df

frames_wb = []
for ind, label in WB_INDICATORS.items():
    print(f"  WB {ind} ({label}) ...", end="  ", flush=True)
    try:
        df = fetch_wb(ind)
        df["indicator_label"] = label
        frames_wb.append(df)
        print(f"{len(df):,} records")
    except Exception as exc:
        print(f"ERROR — {exc}")

df_wb_raw = pd.concat(frames_wb, ignore_index=True)
print(f"\nWorld Bank raw: {len(df_wb_raw):,} rows | "
      f"{df_wb_raw['indicator_id'].nunique()} indicators")
df_wb_raw.head(3)


  WB SH.HIV.INCD.ZS (hiv_incidence_per_1000) ...  120 records
  WB SH.HIV.ARTC.ZS (art_coverage_wb_pct) ...  120 records
  WB SH.XPD.CHEX.GD.ZS (health_exp_pct_gdp) ...  120 records
  WB SH.XPD.CHEX.PC.CD (health_exp_per_cap_usd) ...  120 records
  WB SH.MED.NUMW.P3 (nurses_per_1000) ...  120 records
  WB SH.MED.PHYS.ZS (physicians_per_1000) ...  120 records
  WB SP.POP.TOTL (population_total) ...  120 records
  WB NY.GDP.PCAP.CD (gdp_per_cap_usd) ...  120 records

World Bank raw: 960 rows | 8 indicators


,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value,indicator_id,indicator_label
0,ETH,2023,0.13,,,1,SH.HIV.INCD.ZS,"Incidence of HIV, ages 15-49 (per 1,000 uninfe...",ET,Ethiopia,SH.HIV.INCD.ZS,hiv_incidence_per_1000
1,ETH,2022,0.14,,,1,SH.HIV.INCD.ZS,"Incidence of HIV, ages 15-49 (per 1,000 uninfe...",ET,Ethiopia,SH.HIV.INCD.ZS,hiv_incidence_per_1000
2,ETH,2021,0.19,,,1,SH.HIV.INCD.ZS,"Incidence of HIV, ages 15-49 (per 1,000 uninfe...",ET,Ethiopia,SH.HIV.INCD.ZS,hiv_incidence_per_1000


## C. Source 3: UNAIDS Estimates 2026

**URL:** `https://aidsinfo.unaids.org/public/documents/Estimates_2026_en.zip`  
**Format:** ZIP containing `Estimates_2026_en.csv`  
**Auth:** None  
**Release policy:** pinned to 2026 for reproducible reruns

The source CSV is approximately 348 MB uncompressed, so it is read in chunks. Only five countries, four indicators and 2000–2023 are retained.

| UNAIDS indicator code | Normalized metric |
|---|---|
| `PLWH` | `unaids_plhiv` |
| `NEW_INFECTIONS` | `unaids_new_infections` |
| `AIDS_DEATHS` | `unaids_aids_deaths` |
| `AIDS_MORTALITY_1000_POP` | `unaids_aids_mortality_per_1000` |

Central, lower and upper estimates are preserved. This replaces the non-downloadable UNAIDS Estimates 2026 chart CSVs.

In [8]:
import zipfile

UNAIDS_RELEASE_YEAR = 2026
UNAIDS_URL = (
    "https://aidsinfo.unaids.org/public/documents/"
    f"Estimates_{UNAIDS_RELEASE_YEAR}_en.zip"
)
UNAIDS_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; KenyaHealthPipeline/2.0)",
    "Referer": "https://aidsinfo.unaids.org/dataset",
}
UNAIDS_AREA_TO_ISO3 = {
    "Kenya": "KEN",
    "Uganda": "UGA",
    "United Republic of Tanzania": "TZA",
    "Ethiopia": "ETH",
    "Rwanda": "RWA",
}
UNAIDS_METRICS = {
    "PLWH": "unaids_plhiv",
    "NEW_INFECTIONS": "unaids_new_infections",
    "AIDS_DEATHS": "unaids_hiv_related_deaths",
    "AIDS_MORTALITY_1000_POP": "unaids_aids_mortality_per_1000",
}
CENTRAL_SUBGROUP = {
    "PLWH": "ESTIMATE_ALL_AGES",
    "NEW_INFECTIONS": "ESTIMATE_ALL_AGES",
    "AIDS_DEATHS": "ESTIMATE_ALL_AGES",
    "AIDS_MORTALITY_1000_POP": "ESTIMATE_TOTAL",
}
LOW_SUBGROUP = {
    "PLWH": "LOW_ESTIMATE_ALL_AGES",
    "NEW_INFECTIONS": "LOW_ESTIMATE_ALL_AGES",
    "AIDS_DEATHS": "LOW_ESTIMATE_ALL_AGES",
    "AIDS_MORTALITY_1000_POP": "LOW_ESTIMATE_TOTAL",
}
HIGH_SUBGROUP = {
    "PLWH": "HIGH_ESTIMATE_ALL_AGES",
    "NEW_INFECTIONS": "HIGH_ESTIMATE_ALL_AGES",
    "AIDS_DEATHS": "HIGH_ESTIMATE_ALL_AGES",
    "AIDS_MORTALITY_1000_POP": "HIGH_ESTIMATE_TOTAL",
}

print(f"Downloading UNAIDS Estimates {UNAIDS_RELEASE_YEAR} ...")
r = requests.get(UNAIDS_URL, headers=UNAIDS_HEADERS, timeout=180)
r.raise_for_status()
if not r.content.startswith(b"PK"):
    raise ValueError("UNAIDS response is not a ZIP archive")

selected_chunks = []
with zipfile.ZipFile(io.BytesIO(r.content)) as archive:
    csv_members = [name for name in archive.namelist() if name.lower().endswith(".csv")]
    if len(csv_members) != 1:
        raise ValueError(f"Expected one UNAIDS CSV, found: {csv_members}")
    with archive.open(csv_members[0]) as csv_file:
        for chunk in pd.read_csv(csv_file, chunksize=200_000, low_memory=False):
            mask = (
                chunk["Area"].isin(UNAIDS_AREA_TO_ISO3)
                & chunk["Indicator_GId"].isin(UNAIDS_METRICS)
                & chunk["Time Period"].between(YEAR_START, YEAR_END)
                & chunk["Subgroup_Val_GId"].isin(
                    set(CENTRAL_SUBGROUP.values())
                    | set(LOW_SUBGROUP.values())
                    | set(HIGH_SUBGROUP.values())
                )
            )
            if mask.any():
                selected_chunks.append(chunk.loc[mask, [
                    "Indicator", "Indicator_GId", "Unit", "Subgroup_Val_GId",
                    "Area", "Area ID", "Time Period", "Source", "Data value",
                ]])

if not selected_chunks:
    raise ValueError("UNAIDS download contained no rows for the requested scope")

df_unaids_selected = pd.concat(selected_chunks, ignore_index=True)
df_unaids_selected["estimate_kind"] = "other"
for indicator_id in UNAIDS_METRICS:
    df_unaids_selected.loc[
        (df_unaids_selected["Indicator_GId"] == indicator_id)
        & (df_unaids_selected["Subgroup_Val_GId"] == CENTRAL_SUBGROUP[indicator_id]),
        "estimate_kind",
    ] = "value"
    df_unaids_selected.loc[
        (df_unaids_selected["Indicator_GId"] == indicator_id)
        & (df_unaids_selected["Subgroup_Val_GId"] == LOW_SUBGROUP[indicator_id]),
        "estimate_kind",
    ] = "value_low"
    df_unaids_selected.loc[
        (df_unaids_selected["Indicator_GId"] == indicator_id)
        & (df_unaids_selected["Subgroup_Val_GId"] == HIGH_SUBGROUP[indicator_id]),
        "estimate_kind",
    ] = "value_high"

df_unaids_long = (
    df_unaids_selected
    .pivot_table(
        index=["Area", "Area ID", "Time Period", "Indicator", "Indicator_GId", "Unit", "Source"],
        columns="estimate_kind",
        values="Data value",
        aggfunc="first",
    )
    .reset_index()
    .rename(columns={
        "Area": "entity",
        "Area ID": "area_id",
        "Time Period": "year",
        "Indicator": "metric_label",
        "Indicator_GId": "indicator_id",
        "Unit": "unit",
        "Source": "source",
    })
)
df_unaids_long["country_iso3"] = df_unaids_long["entity"].map(UNAIDS_AREA_TO_ISO3)
df_unaids_long["metric"] = df_unaids_long["indicator_id"].map(UNAIDS_METRICS)
df_unaids_long["source_release"] = UNAIDS_RELEASE_YEAR
for column in ["value", "value_low", "value_high"]:
    if column not in df_unaids_long:
        df_unaids_long[column] = pd.NA

df_unaids_long = df_unaids_long[[
    "entity", "country_iso3", "year", "indicator_id", "metric", "metric_label",
    "unit", "value", "value_low", "value_high", "source", "source_release",
]].sort_values(["country_iso3", "metric", "year"]).reset_index(drop=True)

expected = set(UNAIDS_METRICS.values())
actual = set(df_unaids_long["metric"].dropna().unique())
if actual != expected:
    raise ValueError(f"UNAIDS metric mismatch. Expected {expected}; received {actual}")

print(
    f"UNAIDS normalized: {len(df_unaids_long):,} rows | "
    f"{df_unaids_long['metric'].nunique()} metrics | "
    f"{df_unaids_long['country_iso3'].nunique()} countries"
)
df_unaids_long.head(8)

UNAIDS normalized: 480 rows | 4 metrics | 5 countries


estimate_kind,entity,country_iso3,year,indicator_id,metric,metric_label,unit,value,value_low,value_high,source,source_release
0,Ethiopia,ETH,2000,AIDS_MORTALITY_1000_POP,unaids_aids_mortality_per_1000,AIDS mortality per 1000 population,Rate,1.478436,0.911017,2.428383,UNAIDS_Estimates_,2026
1,Ethiopia,ETH,2001,AIDS_MORTALITY_1000_POP,unaids_aids_mortality_per_1000,AIDS mortality per 1000 population,Rate,1.462550,0.901227,2.402288,UNAIDS_Estimates_,2026
2,Ethiopia,ETH,2002,AIDS_MORTALITY_1000_POP,unaids_aids_mortality_per_1000,AIDS mortality per 1000 population,Rate,1.428486,0.880237,2.346337,UNAIDS_Estimates_,2026
3,Ethiopia,ETH,2003,AIDS_MORTALITY_1000_POP,unaids_aids_mortality_per_1000,AIDS mortality per 1000 population,Rate,1.381348,0.851191,2.268912,UNAIDS_Estimates_,2026
4,Ethiopia,ETH,2004,AIDS_MORTALITY_1000_POP,unaids_aids_mortality_per_1000,AIDS mortality per 1000 population,Rate,1.304709,0.803965,2.143029,UNAIDS_Estimates_,2026
5,Ethiopia,ETH,2005,AIDS_MORTALITY_1000_POP,unaids_aids_mortality_per_1000,AIDS mortality per 1000 population,Rate,1.195162,0.736463,1.963095,UNAIDS_Estimates_,2026
6,Ethiopia,ETH,2006,AIDS_MORTALITY_1000_POP,unaids_aids_mortality_per_1000,AIDS mortality per 1000 population,Rate,1.042819,0.642588,1.712866,UNAIDS_Estimates_,2026
7,Ethiopia,ETH,2007,AIDS_MORTALITY_1000_POP,unaids_aids_mortality_per_1000,AIDS mortality per 1000 population,Rate,0.859403,0.529567,1.411600,UNAIDS_Estimates_,2026


## C. Source 4: The Global Fund Data Service OData v4.2

**Base URL:** `https://fetch.theglobalfund.org/v4.2/odata/`  
**Auth:** None

| Entity | Purpose |
|---|---|
| `Grants` | Kenya grant metadata, recipients, components, dates and cumulative amounts |
| `AllFinancialIndicators` | Transaction-level financial observations |

The grant request expands `implementationPeriods`. Each implementation-period `id` is then used as `AllFinancialIndicators.recordId`. Rows labelled `Disbursement_ReferenceRate` provide transaction amounts in the Global Fund reference currency. Negative adjustments are retained.

In [9]:
from concurrent.futures import ThreadPoolExecutor, as_completed

GF_BASE = "https://fetch.theglobalfund.org/v4.2/odata"
GF_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; KenyaHealthPipeline/2.0)",
    "Accept": "application/json",
}

def fetch_gf_pages(endpoint: str, params: dict | None = None) -> list[dict]:
    """Fetch all pages from a Global Fund OData v4.2 entity."""
    rows = []
    url = f"{GF_BASE}/{endpoint}"
    current_params = params
    while url:
        response = requests.get(url, params=current_params, headers=GF_HEADERS, timeout=120)
        response.raise_for_status()
        payload = response.json()
        rows.extend(payload.get("value", []))
        url = payload.get("@odata.nextLink")
        current_params = None
    return rows

print("Fetching Kenya grants and implementation periods ...")
grant_records = fetch_gf_pages(
    "Grants",
    {
        "$filter": "geography/code eq 'KEN'",
        "$expand": "geography,principalRecipient,activityArea,status,implementationPeriods",
        "$top": 5000,
    },
)
if not grant_records:
    raise ValueError("Global Fund Grants returned no Kenya records")

grant_rows = []
implementation_period_rows = []
for grant in grant_records:
    grant_code = grant.get("code")
    geography = grant.get("geography") or {}
    recipient = grant.get("principalRecipient") or {}
    activity = grant.get("activityArea") or {}
    status = grant.get("status") or {}
    grant_rows.append({
        "grant_id": grant_code,
        "grant_guid": grant.get("id"),
        "country_name": geography.get("name"),
        "country_iso3": geography.get("code"),
        "component": activity.get("name") or "Unknown",
        "recipient_name": recipient.get("name"),
        "recipient_type_id": recipient.get("typeId"),
        "grant_status": status.get("statusName"),
        "start_date": grant.get("periodStartDate"),
        "end_date": grant.get("periodEndDate"),
        "currency": grant.get("currency_ReferenceRate"),
        "total_board_approved_usd": grant.get("totalBoardApprovedAmount_ReferenceRate"),
        "total_signed_usd": grant.get("totalSignedAmount_ReferenceRate"),
        "total_committed_usd": grant.get("totalCommitmentAmount_ReferenceRate"),
        "total_disbursed_usd": grant.get("totalDisbursedAmount_ReferenceRate"),
    })
    for period in grant.get("implementationPeriods") or []:
        implementation_period_rows.append({
            "implementation_period_id": period.get("id"),
            "implementation_period_code": period.get("code"),
            "grant_id": grant_code,
            "country_iso3": geography.get("code"),
            "component": activity.get("name") or "Unknown",
        })

df_gf_grants = pd.DataFrame(grant_rows).drop_duplicates(subset=["grant_id"])
df_gf_implementation_periods = pd.DataFrame(implementation_period_rows).dropna(
    subset=["implementation_period_id"]
).drop_duplicates(subset=["implementation_period_id"])

def fetch_implementation_period_financials(period_id: str) -> list[dict]:
    return fetch_gf_pages(
        "AllFinancialIndicators",
        {"$filter": f"recordId eq {period_id}", "$top": 5000},
    )

print(f"Fetching financials for {len(df_gf_implementation_periods):,} Kenya implementation periods ...")
financial_rows = []
with ThreadPoolExecutor(max_workers=6) as pool:
    futures = {
        pool.submit(fetch_implementation_period_financials, row.implementation_period_id): row
        for row in df_gf_implementation_periods.itertuples(index=False)
    }
    for future in as_completed(futures):
        period = futures[future]
        for record in future.result():
            if record.get("financialDataSet") != "Disbursement_ReferenceRate":
                continue
            transaction_date = pd.to_datetime(record.get("transactionDate"), errors="coerce", utc=True)
            if pd.isna(transaction_date):
                continue
            financial_rows.append({
                "financial_indicator_id": record.get("financialIndicatorId"),
                "implementation_period_id": period.implementation_period_id,
                "implementation_period_code": period.implementation_period_code,
                "grant_id": period.grant_id,
                "country_iso3": period.country_iso3,
                "component": period.component,
                "transaction_date": transaction_date.date().isoformat(),
                "year": int(transaction_date.year),
                "quarter": int(transaction_date.quarter),
                "amount_usd": record.get("actualAmount"),
                "currency": record.get("actualAmountCurrency"),
                "financial_dataset": record.get("financialDataSet"),
            })

df_gf_disb = pd.DataFrame(financial_rows)
if df_gf_disb.empty:
    raise ValueError("Global Fund financial endpoint returned no Kenya disbursement transactions")
df_gf_disb = (
    df_gf_disb
    .drop_duplicates(subset=["financial_indicator_id"])
    .sort_values(["transaction_date", "grant_id"])
    .reset_index(drop=True)
)

print(f"  Kenya grants: {len(df_gf_grants):,}")
print(f"  Implementation periods: {len(df_gf_implementation_periods):,}")
print(f"  Reference-rate disbursement transactions: {len(df_gf_disb):,}")
df_gf_grants.head(3)

Fetching Kenya grants and implementation periods ...
Fetching financials for 47 Kenya implementation periods ...
  Kenya grants: 18
  Implementation periods: 47
  Reference-rate disbursement transactions: 907


,grant_id,grant_guid,country_name,country_iso3,component,recipient_name,recipient_type_id,grant_status,start_date,end_date,currency,total_board_approved_usd,total_signed_usd,total_committed_usd,total_disbursed_usd
0,KEN-607-G08-T,f0e3377c-d2e6-4dd1-b29b-043289e0f4ca,Kenya,KEN,Tuberculosis,The National Treasury & Economic Planning of t...,60a4eb2c-39e0-4dcf-bdd5-bea8163f8c32,Administratively Closed,2008-04-01T00:00:00Z,2010-12-31T00:00:00Z,USD,2935506.00,2935506.00,2935506.00,2935506.00
1,KEN-202-G04-T-00,f50dfadf-8a14-41b9-9614-1a90e9ab500d,Kenya,KEN,Tuberculosis,The National Treasury & Economic Planning of t...,60a4eb2c-39e0-4dcf-bdd5-bea8163f8c32,Administratively Closed,2003-11-01T00:00:00Z,2008-10-31T00:00:00Z,USD,3152080.34,3152080.34,3152080.34,3152080.34
2,KEN-011-G13-M,eb6f427c-fa75-4a83-be41-2cbc30a35474,Kenya,KEN,Malaria,The National Treasury & Economic Planning of t...,60a4eb2c-39e0-4dcf-bdd5-bea8163f8c32,Administratively Closed,2012-01-01T00:00:00Z,2016-12-31T00:00:00Z,USD,81882069.63,81882069.63,81882069.63,81882069.63


## 3. E→L: Load Extracted Data into DuckDB Staging

The normalized source DataFrames land in `stg_*` tables. Source-specific identifiers, uncertainty bounds, release metadata and transaction adjustments are preserved.

In [10]:
def stage(df: pd.DataFrame, table: str) -> None:
    """Register a pandas DataFrame as a DuckDB staging table (replace if exists)."""
    con.execute(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM df")
    n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<32}  {n:>8,} rows")

print("Loading staging layer ...")
stage(df_who_raw,    "stg_who_hiv")
stage(df_wb_raw,     "stg_worldbank")
stage(df_unaids_long,"stg_unaids_hiv")
stage(df_gf_grants,  "stg_gf_grants")
stage(df_gf_disb,    "stg_gf_disbursements")

print("\nDuckDB tables:")
print(con.execute("SHOW TABLES").fetchdf().to_string(index=False))

Loading staging layer ...
  stg_who_hiv                          1,110 rows
  stg_worldbank                          960 rows
  stg_unaids_hiv                         480 rows
  stg_gf_grants                           18 rows
  stg_gf_disbursements                   907 rows

DuckDB tables:
                  name
clean_gf_disbursements
       clean_gf_grants
      clean_unaids_hiv
         clean_who_hiv
       clean_worldbank
           dim_country
         dim_indicator
              dim_time
       fact_gf_funding
fact_health_indicators
        feat_gf_annual
    feat_health_system
      feat_hiv_cascade
  stg_gf_disbursements
         stg_gf_grants
        stg_unaids_hiv
           stg_who_hiv
         stg_worldbank


In [11]:
# Schema inspection for all staging tables
for tbl in ["stg_who_hiv", "stg_worldbank", "stg_unaids_hiv",
            "stg_gf_grants", "stg_gf_disbursements"]:
    print(f"\n{'─'*58}  {tbl}")
    schema = con.execute(f"DESCRIBE {tbl}").fetchdf()
    print(schema[["column_name", "column_type"]].to_string(index=False))
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  Total rows: {n:,}")


──────────────────────────────────────────────────────────  stg_who_hiv
            column_name column_type
                     Id      BIGINT
          IndicatorCode     VARCHAR
         SpatialDimType     VARCHAR
             SpatialDim     VARCHAR
            TimeDimType     VARCHAR
     ParentLocationCode     VARCHAR
         ParentLocation     VARCHAR
               Dim1Type     VARCHAR
                TimeDim      BIGINT
                   Dim1     VARCHAR
               Dim2Type     INTEGER
                   Dim2     INTEGER
               Dim3Type     INTEGER
                   Dim3     INTEGER
      DataSourceDimType     INTEGER
          DataSourceDim     INTEGER
                  Value     VARCHAR
           NumericValue      DOUBLE
                    Low      DOUBLE
                   High      DOUBLE
               Comments     INTEGER
                   Date     VARCHAR
     TimeDimensionValue     VARCHAR
     TimeDimensionBegin     VARCHAR
       TimeDimensionEnd    

## 4. L→T : Data Cleaning & Normalisation

Each staging table is typed, filtered, and standardised into `clean_*` tables:
- All dates cast to `INTEGER` year
- `TRY_CAST` for all numeric fields (bad values become `NULL` rather than crashing)
- Sex dimension normalised from WHO codes (`BTSX`, `MLE`, `FMLE`) to readable labels
- Country ISO3 filtered to East Africa scope


In [12]:
# clean_who_hiv
con.execute("""
CREATE OR REPLACE TABLE clean_who_hiv AS
SELECT
    CAST(TimeDim AS INTEGER)          AS year,
    SpatialDim                        AS country_iso3,
    IndicatorCode                     AS indicator_code,
    indicator_label,
    TRY_CAST(NumericValue AS DOUBLE)  AS value,
    TRY_CAST(Low          AS DOUBLE)  AS value_low,
    TRY_CAST(High         AS DOUBLE)  AS value_high,
    -- Normalise WHO sex dimension codes to readable labels
    COALESCE(
        CASE CAST(Dim1 AS VARCHAR)
            WHEN 'MLE'       THEN 'male'
            WHEN 'SEX_MLE'   THEN 'male'
            WHEN 'FMLE'      THEN 'female'
            WHEN 'SEX_FMLE'  THEN 'female'
            WHEN 'BTSX'      THEN 'both_sexes'
            WHEN 'SEX_BTSX'  THEN 'both_sexes'
        END,
        CASE CAST(Dim2 AS VARCHAR)
            WHEN 'MLE'       THEN 'male'
            WHEN 'SEX_MLE'   THEN 'male'
            WHEN 'FMLE'      THEN 'female'
            WHEN 'SEX_FMLE'  THEN 'female'
            WHEN 'BTSX'      THEN 'both_sexes'
            WHEN 'SEX_BTSX'  THEN 'both_sexes'
        END,
        'both_sexes'
    )                                 AS sex,
    current_timestamp                 AS loaded_at
FROM stg_who_hiv
WHERE
    SpatialDimType = 'COUNTRY'
    AND TRY_CAST(TimeDim AS INTEGER) BETWEEN 2000 AND 2023
    AND TRY_CAST(NumericValue AS DOUBLE) IS NOT NULL
""")
n = con.execute("SELECT COUNT(*) FROM clean_who_hiv").fetchone()[0]
print(f"clean_who_hiv            {n:>8,} rows")

# clean_worldbank
con.execute("""
CREATE OR REPLACE TABLE clean_worldbank AS
SELECT
    CAST(date AS INTEGER)             AS year,
    countryiso3code                   AS country_iso3,
    "country.value"                   AS country_name,
    indicator_id,
    indicator_label,
    TRY_CAST(value AS DOUBLE)         AS value,
    current_timestamp                 AS loaded_at
FROM stg_worldbank
WHERE
    CAST(date AS INTEGER) BETWEEN 2000 AND 2023
    AND value IS NOT NULL
    AND countryiso3code IN ('KEN','UGA','TZA','ETH','RWA')
""")
n = con.execute("SELECT COUNT(*) FROM clean_worldbank").fetchone()[0]
print(f"clean_worldbank          {n:>8,} rows")


clean_who_hiv               1,050 rows
clean_worldbank               839 rows


In [13]:
# clean_unaids_hiv
con.execute("""
CREATE OR REPLACE TABLE clean_unaids_hiv AS
SELECT
    entity                            AS country_name,
    country_iso3,
    CAST(year AS INTEGER)             AS year,
    indicator_id,
    metric,
    metric_label,
    unit,
    TRY_CAST(value AS DOUBLE)         AS value,
    TRY_CAST(value_low AS DOUBLE)     AS value_low,
    TRY_CAST(value_high AS DOUBLE)    AS value_high,
    source,
    CAST(source_release AS INTEGER)   AS source_release,
    current_timestamp                 AS loaded_at
FROM stg_unaids_hiv
WHERE
    CAST(year AS INTEGER) BETWEEN 2000 AND 2023
    AND TRY_CAST(value AS DOUBLE) IS NOT NULL
    AND country_iso3 IS NOT NULL
""")
n = con.execute("SELECT COUNT(*) FROM clean_unaids_hiv").fetchone()[0]
print(f"clean_unaids_hiv         {n:>8,} rows")

print("\nUNAIDS metrics (Kenya):")
print(con.execute("""
    SELECT metric, metric_label, unit, MIN(year) AS first_year, MAX(year) AS last_year
    FROM clean_unaids_hiv
    WHERE country_iso3 = 'KEN'
    GROUP BY ALL ORDER BY metric
""").fetchdf().to_string(index=False))

clean_unaids_hiv              480 rows

UNAIDS metrics (Kenya):
                        metric                       metric_label   unit  first_year  last_year
unaids_aids_mortality_per_1000 AIDS mortality per 1000 population   Rate        2000       2023
     unaids_hiv_related_deaths                AIDS-related deaths Number        2000       2023
         unaids_new_infections                 New HIV Infections Number        2000       2023
                  unaids_plhiv             People living with HIV Number        2000       2023


In [14]:
# clean_gf_grants
con.execute("""
CREATE OR REPLACE TABLE clean_gf_grants AS
SELECT
    grant_id,
    grant_guid,
    country_name,
    country_iso3,
    COALESCE(component, 'Unknown')                    AS component,
    recipient_name,
    recipient_type_id                                AS recipient_type,
    grant_status,
    TRY_CAST(start_date AS DATE)                      AS start_date,
    TRY_CAST(end_date AS DATE)                        AS end_date,
    YEAR(TRY_CAST(start_date AS DATE))                AS start_year,
    TRY_CAST(total_board_approved_usd AS DOUBLE)      AS total_board_approved_usd,
    TRY_CAST(total_signed_usd AS DOUBLE)              AS total_signed_usd,
    TRY_CAST(total_committed_usd AS DOUBLE)           AS total_committed_usd,
    TRY_CAST(total_disbursed_usd AS DOUBLE)           AS total_disbursed_usd,
    CASE WHEN TRY_CAST(total_signed_usd AS DOUBLE) > 0
         THEN ROUND(TRY_CAST(total_disbursed_usd AS DOUBLE)
                    / TRY_CAST(total_signed_usd AS DOUBLE) * 100, 2)
         ELSE NULL END                                AS disbursement_rate_pct,
    currency,
    current_timestamp                                 AS loaded_at
FROM stg_gf_grants
WHERE country_iso3 = 'KEN'
""")
n1 = con.execute("SELECT COUNT(*) FROM clean_gf_grants").fetchone()[0]
print(f"clean_gf_grants          {n1:>8,} rows")

#clean_gf_disbursements
con.execute("""
CREATE OR REPLACE TABLE clean_gf_disbursements AS
SELECT
    financial_indicator_id,
    implementation_period_id,
    implementation_period_code,
    grant_id,
    country_iso3,
    COALESCE(component, 'Unknown')                    AS component,
    TRY_CAST(transaction_date AS DATE)                AS transaction_date,
    CAST(year AS INTEGER)                             AS year,
    CAST(quarter AS INTEGER)                          AS quarter,
    TRY_CAST(amount_usd AS DOUBLE)                    AS amount_usd,
    currency,
    financial_dataset
FROM stg_gf_disbursements
WHERE
    country_iso3 = 'KEN'
    AND year IS NOT NULL
    AND TRY_CAST(amount_usd AS DOUBLE) IS NOT NULL
""")
n2 = con.execute("SELECT COUNT(*) FROM clean_gf_disbursements").fetchone()[0]
print(f"clean_gf_disbursements   {n2:>8,} rows (negative adjustments retained)")

clean_gf_grants                18 rows
clean_gf_disbursements        907 rows (negative adjustments retained)


## 5. Feature Engineering

Computed directly in DuckDB SQL using window functions and CASE logic.

| Feature Table | Derived Metrics |
|---------------|-----------------|
| `feat_hiv_cascade` | Treatment gap, YoY ART coverage Δ, annual HIV-related deaths as a percentage of PLHIV, 95-95-95 first-95 proxy |
| `feat_health_system` | Composite HCW density, health investment intensity, ART coverage Δ since 2010 |
| `feat_gf_annual` | Annual & cumulative Global Fund disbursements by component |


In [15]:
# feat_hiv_cascade
con.execute("""
CREATE OR REPLACE TABLE feat_hiv_cascade AS
WITH cascade_pivot AS (
    SELECT
        year, country_iso3,
        MAX(CASE WHEN indicator_code = 'HIV_0000000001' AND sex = 'both_sexes'
                 THEN value END)                        AS plhiv,
        MAX(CASE WHEN indicator_code = 'MDG_0000000029' AND sex = 'both_sexes'
                 THEN value END)                        AS hiv_prev_pct,
        MAX(CASE WHEN indicator_code = 'HIV_0000000009' AND sex = 'both_sexes'
                 THEN value END)                        AS on_art,
        MAX(CASE WHEN indicator_code = 'HIV_ARTCOVERAGE' AND sex = 'both_sexes'
                 THEN value END)                        AS art_coverage_pct,
        MAX(CASE WHEN indicator_code = 'HIV_0000000026' AND sex = 'both_sexes'
                 THEN value END)                        AS new_infections,
        MAX(CASE WHEN indicator_code = 'HIV_0000000006' AND sex = 'both_sexes'
                 THEN value END)                        AS hiv_related_deaths,
        MAX(CASE WHEN indicator_code = 'WHOSIS_000001' AND sex = 'both_sexes'
                 THEN value END)                        AS life_expectancy
    FROM clean_who_hiv
    GROUP BY year, country_iso3
)
SELECT
    year, country_iso3,
    ROUND(plhiv)                                        AS plhiv,
    hiv_prev_pct,
    ROUND(on_art)                                       AS on_art,
    art_coverage_pct,
    ROUND(new_infections)                               AS new_infections,
    ROUND(hiv_related_deaths)                           AS hiv_related_deaths,
    life_expectancy,
    -- Coverage-based estimate; source inputs are rounded and modelled.
    CASE WHEN plhiv IS NOT NULL AND art_coverage_pct BETWEEN 0 AND 100
         THEN ROUND(plhiv * (1 - art_coverage_pct / 100))
         ELSE NULL END                                  AS treatment_gap,
    art_coverage_pct
        - LAG(art_coverage_pct) OVER (PARTITION BY country_iso3 ORDER BY year)
                                                        AS art_cov_yoy_pp,
    new_infections
        - LAG(new_infections) OVER (PARTITION BY country_iso3 ORDER BY year)
                                                        AS infections_yoy_delta,
    -- Annual HIV-related deaths as a percentage of estimated PLHIV.
    CASE WHEN plhiv > 0 AND hiv_related_deaths IS NOT NULL
         THEN ROUND(hiv_related_deaths / plhiv * 100, 3)
         ELSE NULL END                                  AS hiv_deaths_pct_plhiv,
    -- Analytical proxy only; not an official first-95 measure.
    LEAST(COALESCE(art_coverage_pct, 0) * 1.08, 100.0) AS est_diagnosed_pct_proxy
FROM cascade_pivot
ORDER BY country_iso3, year
""")

# Fail fast on impossible coverage-based results.
invalid_cascade_rows = con.execute("""
    SELECT COUNT(*)
    FROM feat_hiv_cascade
    WHERE art_coverage_pct NOT BETWEEN 0 AND 100
       OR treatment_gap < 0
""").fetchone()[0]
if invalid_cascade_rows:
    raise ValueError(f"Invalid cascade rows detected: {invalid_cascade_rows}")

n = con.execute("SELECT COUNT(*) FROM feat_hiv_cascade").fetchone()[0]
print(f"feat_hiv_cascade         {n:>8,} rows")

feat_hiv_cascade              120 rows


In [16]:
# feat_health_system
con.execute("""
CREATE OR REPLACE TABLE feat_health_system AS
WITH wb_pivot AS (
    SELECT
        year, country_iso3,
        MAX(CASE WHEN indicator_label = 'hiv_incidence_per_1000'   THEN value END) AS hiv_incidence,
        MAX(CASE WHEN indicator_label = 'art_coverage_wb_pct'       THEN value END) AS art_coverage_pct,
        MAX(CASE WHEN indicator_label = 'health_exp_pct_gdp'        THEN value END) AS health_exp_pct_gdp,
        MAX(CASE WHEN indicator_label = 'health_exp_per_cap_usd'    THEN value END) AS health_exp_per_cap,
        MAX(CASE WHEN indicator_label = 'nurses_per_1000'           THEN value END) AS nurses_per_1000,
        MAX(CASE WHEN indicator_label = 'physicians_per_1000'       THEN value END) AS physicians_per_1000,
        MAX(CASE WHEN indicator_label = 'population_total'          THEN value END) AS population,
        MAX(CASE WHEN indicator_label = 'gdp_per_cap_usd'          THEN value END) AS gdp_per_cap
    FROM clean_worldbank
    GROUP BY year, country_iso3
)
SELECT
    year, country_iso3,
    ROUND(population)                                   AS population,
    gdp_per_cap,
    health_exp_pct_gdp,
    health_exp_per_cap,
    physicians_per_1000,
    nurses_per_1000,
    -- Composite health workforce density (WHO threshold = 4.45 per 1,000)
    ROUND(COALESCE(physicians_per_1000, 0)
          + COALESCE(nurses_per_1000, 0), 4)            AS total_hcw_per_1000,
    -- Health investment intensity: health spend per unit of income
    CASE WHEN gdp_per_cap > 0
         THEN ROUND(health_exp_per_cap / gdp_per_cap * 1000, 3)
         ELSE NULL END                                  AS health_invest_intensity,
    hiv_incidence,
    art_coverage_pct,
    -- ART coverage gain relative to country's own 2010 baseline
    art_coverage_pct
        - FIRST_VALUE(art_coverage_pct) OVER (
              PARTITION BY country_iso3
              ORDER BY CASE WHEN year >= 2010 THEN year ELSE 9999 END
              ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
                                                        AS art_cov_delta_since_2010
FROM wb_pivot
ORDER BY country_iso3, year
""")
n = con.execute("SELECT COUNT(*) FROM feat_health_system").fetchone()[0]
print(f"feat_health_system       {n:>8,} rows")

# feat_gf_annual
con.execute("""
CREATE OR REPLACE TABLE feat_gf_annual AS
SELECT
    year, component,
    SUM(amount_usd)                                     AS annual_disbursed_usd,
    COUNT(DISTINCT grant_id)                            AS grant_count,
    SUM(SUM(amount_usd)) OVER (
        PARTITION BY component ORDER BY year)           AS cumulative_usd
FROM clean_gf_disbursements
WHERE year BETWEEN 2002 AND 2023
GROUP BY year, component
ORDER BY year, component
""")
n = con.execute("SELECT COUNT(*) FROM feat_gf_annual").fetchone()[0]
print(f"feat_gf_annual           {n:>8,} rows")


feat_health_system            120 rows
feat_gf_annual                 60 rows


## 6. Data Model: Star Schema

```
dim_country ──┐
dim_time    ──┼──► fact_health_indicators   (WHO GHO + World Bank unified)
dim_indicator ┘

dim_country ──┐
dim_time    ──┴──► fact_gf_funding          (Global Fund disbursements + grant metadata)
```

Dimension tables provide human-readable context; fact tables carry the measurements.


In [17]:
# dim_country
con.execute("""
CREATE OR REPLACE TABLE dim_country AS
SELECT DISTINCT
    country_iso3,
    CASE country_iso3
        WHEN 'KEN' THEN 'Kenya'
        WHEN 'UGA' THEN 'Uganda'
        WHEN 'TZA' THEN 'Tanzania'
        WHEN 'ETH' THEN 'Ethiopia'
        WHEN 'RWA' THEN 'Rwanda'
        ELSE country_iso3
    END AS country_name,
    'East Africa' AS sub_region,
    country_iso3 = 'KEN' AS is_focus
FROM (
    SELECT country_iso3 FROM clean_who_hiv
    UNION SELECT country_iso3 FROM clean_worldbank
    UNION SELECT country_iso3 FROM clean_unaids_hiv
)
""")

# dim_time
con.execute("""
CREATE OR REPLACE TABLE dim_time AS
SELECT y AS year, y - 2000 AS years_from_2000,
    CASE WHEN y < 2004 THEN 'Pre-ART Scale-up'
         WHEN y BETWEEN 2004 AND 2012 THEN 'ART Scale-up'
         WHEN y BETWEEN 2013 AND 2019 THEN 'Test & Treat'
         ELSE 'COVID & Recovery' END AS policy_era,
    CASE y WHEN 2003 THEN 'PEPFAR launched'
           WHEN 2014 THEN '90-90-90 targets set'
           WHEN 2021 THEN '95-95-95 targets set' END AS hiv_milestone
FROM RANGE(2000, 2024) t(y)
""")

# dim_indicator
con.execute("""
CREATE OR REPLACE TABLE dim_indicator AS
SELECT indicator_code AS indicator_id, indicator_label, 'WHO GHO' AS source, 'HIV/AIDS' AS domain
FROM clean_who_hiv GROUP BY ALL
UNION ALL
SELECT indicator_id, indicator_label, 'World Bank',
    CASE WHEN indicator_label LIKE '%hiv%' OR indicator_label LIKE '%art%' THEN 'HIV/AIDS'
         WHEN indicator_label LIKE '%nurse%' OR indicator_label LIKE '%physician%' THEN 'Health Workforce'
         WHEN indicator_label LIKE '%exp%' THEN 'Health Financing'
         ELSE 'Demographics' END
FROM clean_worldbank GROUP BY ALL
UNION ALL
SELECT indicator_id, metric_label, 'UNAIDS Estimates', 'HIV/AIDS'
FROM clean_unaids_hiv GROUP BY ALL
""")

# fact_health_indicators
con.execute("""
CREATE OR REPLACE TABLE fact_health_indicators AS
SELECT year, country_iso3, indicator_code AS indicator_id, indicator_label,
       value, value_low, value_high, sex, 'WHO GHO' AS source
FROM clean_who_hiv
UNION ALL
SELECT year, country_iso3, indicator_id, indicator_label,
       value, NULL, NULL, 'both_sexes', 'World Bank'
FROM clean_worldbank
UNION ALL
SELECT year, country_iso3, indicator_id, metric_label,
       value, value_low, value_high, 'both_sexes', 'UNAIDS Estimates'
FROM clean_unaids_hiv
""")

# fact_gf_funding
con.execute("""
CREATE OR REPLACE TABLE fact_gf_funding AS
SELECT d.year, d.quarter, d.transaction_date, d.country_iso3,
       d.component, d.grant_id, d.amount_usd,
       g.recipient_name, g.recipient_type, g.grant_status,
       g.total_signed_usd, g.total_disbursed_usd, g.disbursement_rate_pct
FROM clean_gf_disbursements d
LEFT JOIN clean_gf_grants g USING (grant_id)
""")

print("Star schema tables:")
for tbl in ["dim_country", "dim_time", "dim_indicator", "fact_health_indicators", "fact_gf_funding"]:
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<30}  {n:>8,} rows")

Star schema tables:
  dim_country                            5 rows
  dim_time                              24 rows
  dim_indicator                         19 rows
  fact_health_indicators             2,369 rows
  fact_gf_funding                      907 rows


In [18]:
# Full data lineage summary
lineage = con.execute("""
    SELECT table_name AS table,
           estimated_size AS approx_bytes
    FROM duckdb_tables()
    ORDER BY table_name
""").fetchdf()

all_tbls = con.execute("SHOW TABLES").fetchdf()["name"].tolist()
rows_list = []
for t in sorted(all_tbls):
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    layer = ("staging" if t.startswith("stg_")
             else "clean"   if t.startswith("clean_")
             else "feature" if t.startswith("feat_")
             else "dim"     if t.startswith("dim_")
             else "fact"    if t.startswith("fact_")
             else "other")
    rows_list.append({"table": t, "layer": layer, "rows": n})

df_lineage = pd.DataFrame(rows_list)
print(df_lineage.to_string(index=False))


                 table   layer  rows
clean_gf_disbursements   clean   907
       clean_gf_grants   clean    18
      clean_unaids_hiv   clean   480
         clean_who_hiv   clean  1050
       clean_worldbank   clean   839
           dim_country     dim     5
         dim_indicator     dim    19
              dim_time     dim    24
       fact_gf_funding    fact   907
fact_health_indicators    fact  2369
        feat_gf_annual feature    60
    feat_health_system feature   120
      feat_hiv_cascade feature   120
  stg_gf_disbursements staging   907
         stg_gf_grants staging    18
        stg_unaids_hiv staging   480
           stg_who_hiv staging  1110
         stg_worldbank staging   960


## 7. Analytics, Query Layer

Pull analysis-ready DataFrames from the star schema and feature tables.

In [19]:
# 1. Kenya HIV cascade time series
df_cascade = con.execute("""
    SELECT c.year, t.policy_era, t.hiv_milestone,
           c.plhiv, c.on_art, c.art_coverage_pct,
           c.new_infections, c.hiv_related_deaths,
           c.hiv_prev_pct, c.treatment_gap,
           c.art_cov_yoy_pp, c.infections_yoy_delta,
           c.hiv_deaths_pct_plhiv, c.life_expectancy
    FROM feat_hiv_cascade c
    JOIN dim_time t ON c.year = t.year
    WHERE c.country_iso3 = 'KEN' AND c.year >= 2005
    ORDER BY c.year
""").fetchdf()

# 2. Regional health system metrics
df_regional = con.execute("""
    SELECT hs.year, hs.country_iso3, dc.country_name,
           hs.art_coverage_pct, hs.hiv_incidence,
           hs.health_exp_per_cap, hs.health_exp_pct_gdp,
           hs.total_hcw_per_1000, hs.gdp_per_cap
    FROM feat_health_system hs
    JOIN dim_country dc ON hs.country_iso3 = dc.country_iso3
    WHERE hs.year >= 2010
    ORDER BY hs.country_iso3, hs.year
""").fetchdf()

# 3. HIV incidence in all countries
df_incidence = con.execute("""
    SELECT f.year, f.country_iso3, dc.country_name,
           f.value AS hiv_incidence_per_1000, t.policy_era
    FROM fact_health_indicators f
    JOIN dim_time t     ON f.year = t.year
    JOIN dim_country dc ON f.country_iso3 = dc.country_iso3
    WHERE f.indicator_label = 'hiv_incidence_per_1000'
      AND f.year >= 2004
    ORDER BY f.country_iso3, f.year
""").fetchdf()

# 4. Global Fund disbursements
df_gf_viz = con.execute("""
    SELECT year, component,
           annual_disbursed_usd / 1e6  AS annual_usd_m,
           cumulative_usd       / 1e6  AS cumulative_usd_m,
           grant_count
    FROM feat_gf_annual
    WHERE annual_disbursed_usd > 0
    ORDER BY year, component
""").fetchdf()

# 5. Health expenditure vs outcomes 
df_scatter = con.execute("""
    SELECT hs.year, hs.country_iso3, dc.country_name,
           hs.health_exp_per_cap, hs.art_coverage_pct,
           hs.hiv_incidence, hs.total_hcw_per_1000, hs.gdp_per_cap
    FROM feat_health_system hs
    JOIN dim_country dc ON hs.country_iso3 = dc.country_iso3
    WHERE hs.art_coverage_pct IS NOT NULL
      AND hs.health_exp_per_cap IS NOT NULL
      AND hs.year >= 2005
    ORDER BY hs.country_iso3, hs.year
""").fetchdf()

print(f"df_cascade   {len(df_cascade):>5,} rows")
print(f"df_regional  {len(df_regional):>5,} rows")
print(f"df_incidence {len(df_incidence):>5,} rows")
print(f"df_gf_viz    {len(df_gf_viz):>5,} rows")
print(f"df_scatter   {len(df_scatter):>5,} rows")


df_cascade      19 rows
df_regional     70 rows
df_incidence   100 rows
df_gf_viz       60 rows
df_scatter      95 rows


In [ ]:
# Snapshot summary: Kenya latest coherent HIV data point
required_hiv_fields = [
    "plhiv", "on_art", "art_coverage_pct", "new_infections", "hiv_related_deaths"
]
latest = df_cascade.dropna(subset=required_hiv_fields).iloc[-1]
life_latest = df_cascade.dropna(subset=["life_expectancy"]).iloc[-1]

if not 0 <= latest["art_coverage_pct"] <= 100:
    raise ValueError(f"Impossible ART coverage: {latest['art_coverage_pct']}")
if latest["treatment_gap"] < 0:
    raise ValueError(f"Impossible negative treatment gap: {latest['treatment_gap']}")

print("=" * 62)
print("  KENYA HIV PROGRAMME SNAPSHOT (WHO GHO, central estimates)")
print("=" * 62)
print(f"  HIV indicator year             : {int(latest['year'])}")
print(f"  PLHIV (estimated)              : {latest['plhiv']:>12,.0f}")
print(f"  People receiving ART          : {latest['on_art']:>12,.0f}")
print(f"  Coverage-based treatment gap  : {latest['treatment_gap']:>12,.0f}")
print(f"  ART coverage                   : {latest['art_coverage_pct']:>11.1f}%")
print(f"  New HIV infections            : {latest['new_infections']:>12,.0f}")
print(f"  HIV-related deaths            : {latest['hiv_related_deaths']:>12,.0f}")
print(
    f"  Life expectancy ({int(life_latest['year'])})        : "
    f"{life_latest['life_expectancy']:>11.1f} yrs"
)
print(
    f"  HIV deaths / PLHIV            : "
    f"{latest['hiv_deaths_pct_plhiv']:>11.3f}%"
)

first = df_cascade.dropna(subset=["art_coverage_pct"]).iloc[0]
pp_gain = latest["art_coverage_pct"] - first["art_coverage_pct"]
target_distance = 95 - latest["art_coverage_pct"]
print(
    f"\n  ART coverage gain since {int(first['year'])}: "
    f"{pp_gain:+.1f} percentage points"
)
if target_distance >= 0:
    print(f"  Distance to 95% benchmark: {target_distance:.1f} pp remaining")
else:
    print(f"  Distance to 95% benchmark: {-target_distance:.1f} pp above benchmark")

# Sum source transactions directly so negative adjustments remain included.
gf_hiv_total = con.execute("""
    SELECT COALESCE(SUM(amount_usd), 0) / 1e6
    FROM clean_gf_disbursements
    WHERE component = 'HIV'
""").fetchone()[0]
print(
    f"\n  Global Fund net HIV disbursements "
    f"(exact HIV component): USD {gf_hiv_total:,.1f}M"
)

  KENYA HIV PROGRAMME SNAPSHOT (WHO GHO, central estimates)
  HIV indicator year             : 2023
  PLHIV (estimated)              :    1,500,000
  People receiving ART          :    1,321,220
  Coverage-based treatment gap  :      180,000
  ART coverage                   :        88.0%
  New HIV infections            :       18,000
  HIV-related deaths            :       22,000
  Life expectancy (2021)        :        66.8 yrs
  HIV deaths / PLHIV            :       1.467%

  ART coverage gain since 2005: +84.0 percentage points
  Distance to 95% benchmark: 7.0 pp remaining

  Global Fund net HIV disbursements (exact HIV component): USD 1,219.5M


## 8. Visualizations with Altair

Five charts covering HIV burden trajectory, treatment cascade progress, regional benchmarking, and financing.

In [ ]:
# Shared colour palettes
COUNTRY_PAL = alt.Scale(
    domain=["KEN","UGA","TZA","ETH","RWA"],
    range=["#E74C3C","#3498DB","#2ECC71","#F39C12","#9B59B6"]
)
ERA_PAL = alt.Scale(
    domain=["Pre-ART Scale-up","ART Scale-up","Test & Treat","COVID & Recovery"],
    range=["#FADBD8","#FDEBD0","#D5F5E3","#D6EAF8"]
)
BASE_TITLE = dict(fontSize=14, subtitleFontSize=11, color="#2C3E50", subtitleColor="#7F8C8D", anchor="start")


In [32]:
# Chart 1. Kenya HIV Burden: PLHIV, On ART, AIDS Deaths
df_c1 = (
    df_cascade[["year","plhiv","on_art","hiv_related_deaths"]]
    .melt(id_vars="year", var_name="metric", value_name="count")
    .dropna()
)
df_c1["label"] = df_c1["metric"].map({
    "plhiv":       "People Living with HIV",
    "on_art":      "On ART",
    "hiv_related_deaths": "AIDS-Related Deaths",
})

burden_pal = alt.Scale(
    domain=["People Living with HIV","On ART","AIDS-Related Deaths"],
    range=["#E74C3C","#27AE60","#95A5A6"]
)

chart1 = (
    alt.Chart(df_c1)
    .mark_line(point=alt.OverlayMarkDef(size=55, filled=True), strokeWidth=2.5)
    .encode(
        x=alt.X("year:O", title="Year",
                 axis=alt.Axis(labelAngle=-45, labelFontSize=11)),
        y=alt.Y("count:Q", title="Estimated Count",
                 axis=alt.Axis(format="~s", labelFontSize=11)),
        color=alt.Color("label:N", scale=burden_pal,
                         legend=alt.Legend(title=None, orient="top-right",
                                           labelFontSize=11)),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("label:N", title="Metric"),
            alt.Tooltip("count:Q", title="Count", format=",.0f"),
        ],
    )
    .properties(
        width=680, height=320,
        title=alt.TitleParams(
            text="Kenya HIV Burden: PLHIV, ART Coverage & AIDS Deaths (2005–2023)",
            subtitle="Source: WHO Global Health Observatory OData API.  95% confidence bounds available in raw data",
            **BASE_TITLE,
        ),
    )
    .interactive()
)
chart1


alt.Chart(...)

In [33]:
# Chart 2. East Africa ART Coverage — Regional Comparison
df_c2 = df_regional.dropna(subset=["art_coverage_pct"]).copy()
df_c2_latest = df_c2.sort_values("year").groupby("country_iso3").last().reset_index()

lines = (
    alt.Chart(df_c2)
    .mark_line(strokeWidth=2.5)
    .encode(
        x=alt.X("year:O", title="Year",
                 axis=alt.Axis(labelAngle=-45, labelFontSize=11)),
        y=alt.Y("art_coverage_pct:Q", title="ART Coverage (% of PLHIV)",
                 scale=alt.Scale(domain=[0, 100]),
                 axis=alt.Axis(labelFontSize=11)),
        color=alt.Color("country_iso3:N", scale=COUNTRY_PAL,
                         legend=alt.Legend(title="Country")),
        strokeDash=alt.condition(
            alt.datum["country_iso3"] == "KEN",
            alt.value([1, 0]),
            alt.value([5, 3]),
        ),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("country_name:N", title="Country"),
            alt.Tooltip("art_coverage_pct:Q", title="ART Coverage (%)", format=".1f"),
        ],
    )
)

points = lines.mark_point(size=55, filled=True)

target = (
    alt.Chart(pd.DataFrame({"y": [95], "label": ["95% UNAIDS target"]}))
    .mark_rule(color="#2C3E50", strokeDash=[6, 3], strokeWidth=1.5, opacity=0.7)
    .encode(y="y:Q")
)
target_txt = (
    alt.Chart(pd.DataFrame({"y": [96.5], "x": [0], "t": ["← UNAIDS 95% target"]}))
    .mark_text(align="left", fontSize=10, color="#2C3E50", fontStyle="italic")
    .encode(y="y:Q", x=alt.value(4), text="t:N")
)

chart2 = (
    (lines + points + target + target_txt)
    .properties(
        width=680, height=320,
        title=alt.TitleParams(
            text="ART Coverage Among PLHIV: East Africa Regional Comparison (2010–2023)",
            subtitle="Source: World Bank Open Data API.  Kenya = solid line, peers = dashed",
            **BASE_TITLE,
        ),
    )
    .interactive()
)
chart2


alt.LayerChart(...)

In [34]:
# Chart 3. Kenya HIV Incidence by Programme Era
df_c3 = df_incidence[df_incidence["country_iso3"] == "KEN"].copy()

era_df = pd.DataFrame([
    {"start": 2004, "end": 2013, "era": "ART Scale-up"},
    {"start": 2013, "end": 2020, "era": "Test & Treat"},
    {"start": 2020, "end": 2024, "era": "COVID & Recovery"},
])

bands = (
    alt.Chart(era_df)
    .mark_rect(opacity=0.35)
    .encode(
        x=alt.X("start:Q", scale=alt.Scale(domain=[2004, 2024])),
        x2="end:Q",
        color=alt.Color("era:N", scale=ERA_PAL,
                         legend=alt.Legend(title="Programme Era")),
    )
)

line_inc = (
    alt.Chart(df_c3)
    .mark_line(color="#C0392B", strokeWidth=3)
    .encode(
        x=alt.X("year:Q", title="Year",
                 scale=alt.Scale(domain=[2004, 2024]),
                 axis=alt.Axis(format="d", labelFontSize=11)),
        y=alt.Y("hiv_incidence_per_1000:Q",
                 title="HIV Incidence (per 1,000 uninfected adults 15–49)",
                 axis=alt.Axis(labelFontSize=11)),
        tooltip=[
            alt.Tooltip("year:Q", title="Year", format="d"),
            alt.Tooltip("hiv_incidence_per_1000:Q",
                        title="Incidence per 1,000", format=".3f"),
            alt.Tooltip("policy_era:N", title="Programme Era"),
        ],
    )
)

dots_inc = (
    alt.Chart(df_c3)
    .mark_circle(color="#C0392B", size=75, opacity=0.9)
    .encode(x="year:Q", y="hiv_incidence_per_1000:Q")
)

chart3 = (
    (bands + line_inc + dots_inc)
    .properties(
        width=680, height=320,
        title=alt.TitleParams(
            text="Kenya HIV Incidence Trajectory by Programme Era (2004–2023)",
            subtitle="Source: World Bank Open Data API.  Shaded regions = HIV programme periods",
            **BASE_TITLE,
        ),
    )
    .interactive()
)
chart3


alt.LayerChart(...)

In [31]:
# Chart 4. Health Expenditure vs ART Coverage
df_c4 = df_scatter.dropna(subset=["art_coverage_pct","health_exp_per_cap"]).copy()
df_c4_latest = df_c4.sort_values("year").groupby("country_iso3").last().reset_index()

bubbles = (
    alt.Chart(df_c4)
    .mark_circle(opacity=0.55, stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X("health_exp_per_cap:Q",
                 title="Health Expenditure per Capita (USD, log scale)",
                 scale=alt.Scale(type="log"),
                 axis=alt.Axis(format=",.0f", labelFontSize=11)),
        y=alt.Y("art_coverage_pct:Q",
                 title="ART Coverage (% of PLHIV)",
                 scale=alt.Scale(domain=[0, 100]),
                 axis=alt.Axis(labelFontSize=11)),
        color=alt.Color("country_iso3:N", scale=COUNTRY_PAL,
                         legend=alt.Legend(title="Country")),
        size=alt.Size("year:Q",
                      scale=alt.Scale(range=[25, 220]),
                      legend=alt.Legend(title="Year")),
        tooltip=[
            alt.Tooltip("year:Q", title="Year", format="d"),
            alt.Tooltip("country_name:N", title="Country"),
            alt.Tooltip("health_exp_per_cap:Q",
                        title="Health Exp/Capita (USD)", format=",.0f"),
            alt.Tooltip("art_coverage_pct:Q",
                        title="ART Coverage (%)", format=".1f"),
            alt.Tooltip("total_hcw_per_1000:Q",
                        title="HCW per 1,000", format=".2f"),
            alt.Tooltip("gdp_per_cap:Q",
                        title="GDP per Capita (USD)", format=",.0f"),
        ],
    )
)

country_labels = (
    alt.Chart(df_c4_latest)
    .mark_text(align="left", dx=8, fontSize=10, fontStyle="italic")
    .encode(
        x="health_exp_per_cap:Q",
        y="art_coverage_pct:Q",
        text="country_iso3:N",
        color=alt.Color("country_iso3:N", scale=COUNTRY_PAL, legend=None),
    )
)

chart4 = (
    (bubbles + country_labels)
    .properties(
        width=620, height=390,
        title=alt.TitleParams(
            text="Health Expenditure per Capita vs ART Coverage in East Africa (2005–2022)",
            subtitle="Source: World Bank Open Data API. Circle size scales with year (larger = more recent)",
            **BASE_TITLE,
        ),
    )
    .interactive()
)
chart4


alt.LayerChart(...)

In [36]:
# Chart 5. Global Fund Kenya Disbursements by Component
df_c5 = df_gf_viz[df_gf_viz["annual_usd_m"] > 0].copy()

comp_pal = alt.Scale(
    domain=["HIV","Tuberculosis","Malaria","RSSH","Multicomponent","Other","Unknown"],
    range=["#E74C3C","#8E44AD","#27AE60","#F39C12","#3498DB","#95A5A6","#BDC3C7"]
)

stacked_bars = (
    alt.Chart(df_c5)
    .mark_bar(stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X("year:O", title="Year",
                 axis=alt.Axis(labelAngle=-45, labelFontSize=11)),
        y=alt.Y("annual_usd_m:Q",
                 title="Annual Disbursements (USD Millions)",
                 stack="zero",
                 axis=alt.Axis(labelFontSize=11)),
        color=alt.Color("component:N", scale=comp_pal,
                         legend=alt.Legend(title="Component")),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("component:N", title="Component"),
            alt.Tooltip("annual_usd_m:Q", title="Disbursed (USD M)", format=",.2f"),
            alt.Tooltip("grant_count:Q", title="Active Grants"),
        ],
    )
)

# Cumulative HIV investment as a dashed overlay
df_hiv_cum = df_c5[df_c5["component"] == "HIV"].copy()
cum_line = (
    alt.Chart(df_hiv_cum)
    .mark_line(color="#2C3E50", strokeWidth=2.2, strokeDash=[5, 3])
    .encode(
        x="year:O",
        y=alt.Y("cumulative_usd_m:Q",
                 axis=alt.Axis(title="Cumulative HIV (USD M)",
                               titleColor="#2C3E50", labelFontSize=10)),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("cumulative_usd_m:Q",
                        title="Cumulative HIV (USD M)", format=",.1f"),
        ],
    )
)

chart5 = (
    alt.layer(stacked_bars, cum_line)
    .resolve_scale(y="independent")
    .properties(
        width=680, height=340,
        title=alt.TitleParams(
            text="Kenya: Global Fund Disbursements by Component (2002–2023)",
            subtitle="Source: The Global Fund Data Service OData API.  Dashed = cumulative HIV disbursements",
            **BASE_TITLE,
        ),
    )
    .interactive()
)
chart5


alt.LayerChart(...)

## 9. Key Analytical Insights

### A. Kenya's HIV treatment coverage expanded sharply between 2005 and 2023

WHO estimates show ART coverage among people living with HIV rising from approximately 4% in 2005 to 88% in 2023, an increase of 84 percentage points. Over the same period, the estimated number of people receiving ART increased from about 54,000 to 1.32 million.

The scale of this increase indicates a major expansion in treatment access over the period covered by the analysis.

### B. Treatment expansion coincided with a large decline in AIDS-related mortality

Estimated AIDS-related deaths fell from approximately 130,000 in 2005 to 22,000 in 2023, a decline of about 83%.

The timing broadly overlaps with the expansion of ART coverage. This analysis is descriptive, however, and should not be interpreted as establishing ART scale-up as the sole cause of the mortality decline.

### C. HIV incidence has also fallen substantially

World Bank estimates show Kenya's HIV incidence among uninfected adults aged 15-49 falling from 4.10 infections per 1,000 in 2004 to 0.56 per 1,000 in 2023.

This represents an approximately 86% decline across the period. The reduction occurs across successive programme eras rather than being concentrated in a single year, although the descriptive analysis does not isolate the effect of individual policies or interventions.

### D. Kenya is among the higher-coverage countries in the regional comparison, but Rwanda records the highest level

World Bank data for 2023 place estimated ART coverage at approximately 93% in Rwanda, 86% in Kenya, 85% in Ethiopia, 82% in Uganda and 79% in Tanzania.

All five countries experienced substantial increases after 2010, suggesting that ART scale-up has been a regional pattern rather than a Kenya-specific development.

Kenya's 2023 ART estimate differs slightly by source: approximately 88% in the WHO series and 86% in the World Bank series. Source-specific estimates should therefore remain separately identified.

### E. Higher overall health expenditure does not translate mechanically into higher ART coverage

Kenya records the highest health expenditure per capita among the five latest observations in the analysis, at roughly USD 85 per person, but Rwanda records higher ART coverage despite lower overall expenditure per capita.

The comparison suggests that national health expenditure alone does not explain ART coverage. Programme design, allocation of resources, service delivery, disease burden and other health-system factors are not isolated by this analysis.

The expenditure variable represents total health spending rather than HIV-specific programme expenditure, so the chart should be interpreted as exploratory rather than causal.

### F. Global Fund financing represents a substantial part of the observed HIV response

The Global Fund series shows more than USD 1.0 billion in HIV-component disbursements to Kenya through 2023 within the visualization window.

Annual disbursements vary considerably across years, indicating that financing has occurred in uneven funding cycles rather than as a constant annual flow.

Negative adjustments and reversals are retained when transactions are aggregated, allowing annual values to represent net financial flows.

## 10. Interpretation Notes

The analysis is primarily descriptive. Relationships between treatment coverage, HIV incidence, mortality, health expenditure and external financing should not be interpreted as causal without additional statistical analysis and controls.


### A. UNAIDS estimates are now a reproducible comparison source
The pipeline downloads the versioned UNAIDS 2026 estimates directly. Central, lower and upper values are preserved for people living with HIV, new infections, AIDS-related deaths and AIDS mortality per 1,000 population.

### B. Source definitions must remain explicit
WHO and UNAIDS estimates may differ because their inputs and modelling methods differ. They are retained as separate source-labelled observations rather than silently combined.

### C. Global Fund annual totals are net transactions
The v4.2 financial API includes negative adjustments and reversals. These are retained so annual aggregation reflects net disbursement flows.

### D. Live-source caveat
WHO, World Bank and Global Fund are live APIs. The UNAIDS release is pinned, but live API values may be revised between executions. The executed DuckDB tables and notebook outputs record the observed run.

---
*Author: Wayne Willis Omondi | Pipeline: DuckDB + pandas ELT | Visualisation: Altair | Data: WHO GHO · World Bank · UNAIDS Estimates 2026 · The Global Fund v4.2*